In [6]:
from data_extraction.db_operations.get_features import execute
results = execute("select id, mbid from artists where mbid is not null")

In [12]:
from data_extraction.utils.rate_limit_handling import RateLimiter, request_with_backoff
import os
lastfm_limiter = RateLimiter(calls_per_second=2)
def get_listeners(mbid):
    LASTFM_API_ENDPOINT = "https://ws.audioscrobbler.com/2.0/"
    LAST_FM_API_KEY = os.getenv("LAST_FM_API_KEY")
    endpoint = "artist.getinfo"
    headers = {"User-Agent": "ArabMusicMap/1.0 (yussef0212@gmail.com)"}
    params = {
        "method": endpoint,
        "mbid": mbid,
        "api_key": LAST_FM_API_KEY,
        "format": "json",
    }

    lastfm_limiter.wait()
    response = request_with_backoff(LASTFM_API_ENDPOINT, params=params, headers=headers)
    return response
    # return int(response['artist']['stats']['listeners'])


In [8]:
from data_extraction.db_operations.get_features import getcon
con = getcon()
def save_listeners(id, listeners):
    listeners = listeners if listeners else 1
    con.execute("update artists set lastfm_listeners = ? where id = ?", [listeners, id])

In [ ]:
import tqdm
fails = []
for item in tqdm.tqdm(results):
    id, mbid = item
    try:
        listeners = get_listeners(mbid) 
    except Exception as e:
        fails.append((item, e))
        print(e)
        continue
    save_listeners(id, listeners)
    